In [1]:
import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, extract_unique_npcis, RF_PARAM_5G

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 10))
# Data filtering
df = filter_dataframe(
    df=df,
    operators=[10],
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq'],
    campaigns=selected_campaigns,
)

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
#df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 42)

unique_npcis = extract_unique_npcis(df['measurements_matrix'])
rf_param = RF_PARAM_5G.RSRQ
n_clusters = 5
random_seed = 42
n_runs = 10

In [ ]:
from scripts.matrix_operations import create_point_matrix
from scripts.utils import dataset_tp_rp_split
from scripts.clustering import train_random_forest
from sklearn.cluster import KMeans
import pandas as pd
import threading
import concurrent.futures
import numpy as np


def train_kmeans(df: pd.DataFrame, n_clusters: int, random_state: int):
    df_features, _ = create_point_matrix(df, unique_npcis, rf_param)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = kmeans.fit_predict(df_features)
    return kmeans, cluster_labels


def process_single_run(df, c, i, random_seeds, unique_npcis, rf_param):
    """Process a single run for a specific cluster size and random seed"""
    random_seed = random_seeds[i]

    # cluster the points
    kmeans, cluster_labels = train_kmeans(df, c, random_seed)
    df_with_clusters = df.copy()
    df_with_clusters['cluster'] = cluster_labels

    # split the dataset
    df_tp, df_rp = dataset_tp_rp_split(df_with_clusters, 0.3, random_seed)

    # train the random forest
    rf = train_random_forest(df_rp, unique_npcis, rf_param, n_estimators=100, random_state=random_seed)

    # predict the cluster
    tp_features, _ = create_point_matrix(df_tp, unique_npcis, rf_param)
    df_tp['predicted'] = rf.predict(tp_features)

    # get the accuracy
    hits = df_tp[df_tp['cluster'] == df_tp['predicted']].shape[0]
    total = df_tp.shape[0]

    # return result
    accuracy = hits / total * 100
    print(f'\rCluster {c} - Run {i+1}/{n_runs} - Accuracy: {accuracy:.2f}%', end='')
    return accuracy


def process_cluster_size(df, c, n_runs, random_seeds, unique_npcis, rf_param):
    """Process all runs for a specific cluster size using threads"""
    results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Submit all runs for this cluster size to the thread pool
        future_to_run = {
            executor.submit(
                process_single_run,
                df, c, i, random_seeds, unique_npcis, rf_param
            ): i for i in range(n_runs)
        }

        # Collect results as they complete
        for future in concurrent.futures.as_completed(future_to_run):
            run_index = future_to_run[future]
            try:
                accuracy = future.result()
                results.append(accuracy)
            except Exception as exc:
                print(f'\nRun {run_index} for cluster {c} generated an exception: {exc}')

    print(f'\nCluster {c} - Average accuracy: {np.mean(results):.2f}%')
    return results


# Main execution
cluster_range = range(1, 10)
n_runs = len(random_seeds)  # Assuming random_seeds is defined elsewhere

# Dictionary to store results
all_results = {c: [] for c in cluster_range}

# Process each cluster size (this could also be parallelized if needed)
for c in cluster_range:
    print(f"Processing cluster size {c}...")
    all_results[c] = process_cluster_size(df, c, n_runs, random_seeds, unique_npcis, rf_param)

# Now all_results contains the accuracy scores for each cluster size and run


Processing cluster size 1...
Cluster 1 - Run 41/1000 - Accuracy: 100.00%